# HireGuard: train the Random Forest + SHAP explainer

Trains the fraud classifier on `hireguard_dataset.csv` (built by
`data-collection/dataset/build_dataset.py`) and exports the two files the
backend loads from `backend/app/ml/artifacts/`:

- `random_forest_model.joblib`
- `shap_explainer.joblib`

Features are computed with the **backend's own** `FeatureExtractor`, cloned
from GitHub, so training uses exactly the features the app computes at
prediction time. If you change `feature_extractor.py`, push the change and
retrain.

Run the cells top to bottom (Runtime > Run all works once the dataset is in
place).

## 1. Install pinned versions
These must match `backend/venv`, or the backend may fail to load the saved files.

In [ ]:
!pip install -q scikit-learn==1.9.1 shap==0.52.0

import sklearn, shap, numpy, pandas, joblib
print("scikit-learn", sklearn.__version__, "| shap", shap.__version__,
      "| numpy", numpy.__version__, "| pandas", pandas.__version__, "| joblib", joblib.__version__)
assert sklearn.__version__ == "1.9.1" and shap.__version__ == "0.52.0", \
    "Version mismatch -- use Runtime > Restart session, then run from this cell again."

## 2. Get the backend feature code from GitHub
Only `backend/app/ml` is imported (no Flask needed). Push any backend changes before running this.

In [ ]:
REPO = "https://github.com/melissandeti22/HireGuard.git"
!rm -rf HireGuard && git clone -q --depth 1 {REPO}
!git -C HireGuard log -1 --format="Using commit %h: %s (%cd)"

import sys
sys.path.insert(0, "HireGuard/backend/app")   # makes `ml` importable without app/__init__.py (Flask)
from ml.feature_extractor import FeatureExtractor
from ml.classifier import FEATURE_ORDER
print("Features:", FEATURE_ORDER)

## 3. Load the dataset
Option A (recommended): put `hireguard_dataset.csv` in Google Drive under `MyDrive/HireGuard/`.
Option B: set `USE_DRIVE = False` and upload it when prompted (~50 MB, slower).

In [ ]:
USE_DRIVE = True
DRIVE_PATH = "/content/drive/MyDrive/HireGuard/hireguard_dataset.csv"

import pandas as pd
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    path = DRIVE_PATH
else:
    from google.colab import files
    uploaded = files.upload()
    path = next(iter(uploaded))

df = pd.read_csv(path, dtype=str).fillna("")
df["fraudulent"] = df["fraudulent"].astype(int)
print(f"{len(df):,} rows")
display(pd.crosstab([df.source, df.split], df.fraudulent, margins=True))
print("label sources:", df.label_source.value_counts().to_dict())
if (df.label_source == "ai_suggested").any():
    print("WARNING: contains AI-suggested labels (built with --use-suggested). "
          "Fine for testing the pipeline; rebuild with verified labels before reporting results.")

## 4. Extract stylometric features
Takes a few minutes. Results are cached to `features.csv` so re-runs are instant (delete it if you rebuild the dataset).

In [ ]:
import os
import numpy as np
from tqdm.auto import tqdm

CACHE = "features.csv"
feats = pd.read_csv(CACHE) if os.path.exists(CACHE) else None
if feats is None or len(feats) != len(df) or list(feats.columns) != FEATURE_ORDER:
    fx = FeatureExtractor()
    rows = [fx.extract(t).to_dict() for t in tqdm(df.text, desc="features")]
    feats = pd.DataFrame(rows)[FEATURE_ORDER].astype(float)
    feats.to_csv(CACHE, index=False)

X = feats[FEATURE_ORDER].to_numpy()
y = df.fraudulent.to_numpy()
groups = df.group.to_numpy()
train, test = (df.split == "train").to_numpy(), (df.split == "test").to_numpy()
print("train", X[train].shape, f"{y[train].mean():.2%} fraud | test", X[test].shape, f"{y[test].mean():.2%} fraud")
display(feats.assign(fraudulent=y).groupby("fraudulent").mean().T.round(3))

## 5. Tune with grouped cross-validation (training split only)
Folds are grouped by employer so near-identical ads never sit on both sides of a fold.
Scored by average precision (PR-AUC), which suits the ~4% fraud rate far better than accuracy.
`class_weight="balanced"` handles the imbalance without synthetic resampling.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold

SEED = 42
param_grid = {
    "n_estimators": [300],
    "max_depth": [None, 12, 24],
    "min_samples_leaf": [1, 3, 10],
    "max_features": ["sqrt", 0.5],
}
search = GridSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=SEED, n_jobs=-1),
    param_grid,
    scoring={"pr_auc": "average_precision", "f1": "f1", "recall": "recall", "precision": "precision"},
    refit="pr_auc",
    cv=StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED),
    n_jobs=-1,
    verbose=1,
)
search.fit(X[train], y[train], groups=groups[train])

cols = ["params", "mean_test_pr_auc", "std_test_pr_auc", "mean_test_f1", "mean_test_recall", "mean_test_precision"]
display(pd.DataFrame(search.cv_results_)[cols].sort_values("mean_test_pr_auc", ascending=False).head(5))
model = search.best_estimator_   # refit on the whole training split
print("Best params:", search.best_params_)

## 6. Evaluate once on the held-out test split
The backend calls `model.predict()`, i.e. a 0.5 probability threshold, so these are the numbers the app will deliver.
Look at the per-source table too: Kenyan postings are what HireGuard is for.

In [ ]:
from sklearn.metrics import (average_precision_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)

proba = model.predict_proba(X[test])[:, 1]
pred = model.predict(X[test])
yt = y[test]
print(classification_report(yt, pred, target_names=["legitimate", "fraudulent"], digits=3))
print("Confusion matrix [[TN FP] [FN TP]]:\n", confusion_matrix(yt, pred))
print(f"PR-AUC {average_precision_score(yt, proba):.3f} | ROC-AUC {roc_auc_score(yt, proba):.3f}")

per_source = []
for src, part in df[test].assign(pred=pred).groupby("source"):
    legit, fraud = part[part.fraudulent == 0], part[part.fraudulent == 1]
    per_source.append({
        "source": src, "rows": len(part), "frauds": len(fraud),
        "recall (frauds caught)": (fraud.pred == 1).mean() if len(fraud) else float("nan"),
        "false-positive rate (legit flagged)": (legit.pred == 1).mean() if len(legit) else float("nan"),
    })
display(pd.DataFrame(per_source).round(3))

## 7. What drives the model

In [ ]:
import matplotlib.pyplot as plt

imp = pd.Series(model.feature_importances_, index=FEATURE_ORDER).sort_values()
imp.plot.barh(title="Random Forest feature importance (impurity)", figsize=(7, 4)); plt.tight_layout(); plt.show()

explainer = shap.TreeExplainer(model)
rng = np.random.default_rng(SEED)
idx = rng.choice(np.where(test)[0], size=min(1000, test.sum()), replace=False)
sv = np.asarray(explainer.shap_values(X[idx]))
sv_fraud = sv[:, :, 1] if sv.ndim == 3 else sv          # SHAP values for the fraudulent class
shap.summary_plot(sv_fraud, X[idx], feature_names=FEATURE_ORDER)

## 8. Save, then check the files load through the backend's own classes

In [ ]:
import json
from ml.classifier import Classifier
from ml.shap_explainer import SHAPExplainer

os.makedirs("artifacts", exist_ok=True)
joblib.dump(model, "artifacts/random_forest_model.joblib")
joblib.dump(explainer, "artifacts/shap_explainer.joblib")

metrics = {
    "best_params": search.best_params_,
    "cv_pr_auc": float(search.best_score_),
    "test": {
        "precision": float(precision_score(yt, pred)), "recall": float(recall_score(yt, pred)),
        "f1": float(f1_score(yt, pred)), "pr_auc": float(average_precision_score(yt, proba)),
        "roc_auc": float(roc_auc_score(yt, proba)),
    },
    "label_sources": df.label_source.value_counts().to_dict(),
    "versions": {"scikit-learn": sklearn.__version__, "shap": shap.__version__},
}
with open("artifacts/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

clf = Classifier("artifacts/random_forest_model.joblib")
exp = SHAPExplainer("artifacts/shap_explainer.joblib")
fx = FeatureExtractor()
scam = ("URGENT HIRING!!! Data entry clerks needed. Earn KES 80,000 per month from home, no experience needed. "
        "Limited slots, act now! Pay a registration fee of KES 1,500 then email recruit.jobs.ke@gmail.com")
vec = fx.extract(scam)
result = clf.predict(vec)
print(result)
assert not result.is_stub, "Backend did not load the model"
if result.label == "fraudulent":
    print(exp.explain(vec, result).plain_language_summary)
print(json.dumps(metrics["test"], indent=2))

## 9. Download and install
Downloads a zip with both `.joblib` files and `metrics.json`. Unzip into
`backend/app/ml/artifacts/` and restart the backend: `/api/classify` will
return `"is_stub_model": false`. Keep `metrics.json` for your results chapter.

In [ ]:
!cd artifacts && zip -q -r ../hireguard_artifacts.zip .
from google.colab import files
files.download("hireguard_artifacts.zip")